# Hub Post-Processing: Line Status Aggregation

This notebook adds line status columns to the final scored hub output.

**Input files:**
1. `hubs_with_complete_data.csv` - Contains line names in `Line_Unique` column
2. `hubs_scored_final.csv` - Final scored hubs output (without line names)
3. Line status CSV with columns: `LineName`, `Year`, `Status`, `Status Id`

**Process:**
1. Extract `group` and `Line_Unique` from `hubs_with_complete_data.csv`
2. Parse and clean `Line_Unique` to extract individual line names
3. Explode to create one row per (group, line) pair
4. Join with status data on line name to get Status Id
5. Aggregate by group and Status Id to count lines per status
6. Merge counts into `hubs_scored_final.csv`

**Output:**
- `hubs_scored_final.csv` with additional columns: `NumLinesStatus_0` through `NumLinesStatus_7`
- Each column counts how many lines in that hub have the corresponding status

## Setup

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from IPython.display import display

# Setup paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
RESULTS_DIR = DATA_DIR / "results"

# Create directories if needed
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Results directory: {RESULTS_DIR}")

## Step 1: Load Hub Data with Line Names

Load `hubs_with_complete_data.csv` which contains the `Line_Unique` column.

In [ ]:
# Load hubs with complete data (includes line names)
hubs_complete_file = RESULTS_DIR / 'hubs_with_complete_data.csv'

# Check alternative locations if not found
if not hubs_complete_file.exists():
    alternative_paths = [
        PROCESSED_DATA_DIR / 'hubs_with_complete_data.csv',
        DATA_DIR / 'hubs_with_complete_data.csv',
        PROJECT_ROOT / 'notebooks' / 'hubs_with_complete_data.csv'
    ]
    
    for path in alternative_paths:
        if path.exists():
            hubs_complete_file = path
            break

if not hubs_complete_file.exists():
    raise FileNotFoundError(
        f"Could not find 'hubs_with_complete_data.csv'.\n"
        f"Please ensure this file exists in one of these locations:\n"
        f"  - {RESULTS_DIR}\n"
        f"  - {PROCESSED_DATA_DIR}\n"
        f"  - {DATA_DIR}"
    )

print(f"Loading hub data with line names from: {hubs_complete_file.name}")
hubs_complete_df = pd.read_csv(hubs_complete_file, encoding='utf-8')

print(f"\nLoaded {len(hubs_complete_df)} hubs")
print(f"Columns: {hubs_complete_df.columns.tolist()[:15]}...")

# Check for required columns
if 'Line_Unique' not in hubs_complete_df.columns:
    raise ValueError("'Line_Unique' column not found in hubs_with_complete_data.csv")
if 'group' not in hubs_complete_df.columns:
    raise ValueError("'group' column not found in hubs_with_complete_data.csv")

print("\n✓ Required columns found: 'group' and 'Line_Unique'")

# Display sample
hubs_complete_df[['group', 'Line_Unique']].head(3)

## Step 2: Load Line Status Data

Load the CSV file containing line status information.

**Expected columns:**
- `LineName`: Name of the transit line (to match with Line_Unique)
- `Year`: Planning year (optional)
- `Status`: Status description (optional)
- `Status Id`: Numeric status identifier (0-7) - **Required**

In [ ]:
# Configure the path to your line status file
line_status_file = DATA_DIR / 'line_status.csv'

# Alternative locations to check
if not line_status_file.exists():
    alternative_paths = [
        PROCESSED_DATA_DIR / 'line_status.csv',
        DATA_DIR / 'lines_with_status.csv',
        PROCESSED_DATA_DIR / 'lines_with_status.csv',
        RESULTS_DIR / 'line_status.csv',
    ]
    
    for path in alternative_paths:
        if path.exists():
            line_status_file = path
            break

if not line_status_file.exists():
    print(f"ERROR: Line status file not found.\n")
    print("Please provide a CSV file with these columns:")
    print("  - LineName: Name of the transit line")
    print("  - Status Id: Numeric status identifier (0-7)")
    print("  - Status: Status description (optional)")
    print("  - Year: Planning year (optional)")
    print("\nSave this file to one of these locations:")
    print(f"  - {DATA_DIR / 'line_status.csv'}")
    print(f"  - {PROCESSED_DATA_DIR / 'line_status.csv'}")
    raise FileNotFoundError("Line status file not found")

print(f"Loading line status from: {line_status_file.name}")
line_status_df = pd.read_csv(line_status_file, encoding='utf-8')

print(f"\nLoaded {len(line_status_df)} line records")
print(f"Columns: {line_status_df.columns.tolist()}")

# Check for required columns
if 'LineName' not in line_status_df.columns:
    raise ValueError("'LineName' column not found in line status file")
if 'Status Id' not in line_status_df.columns:
    raise ValueError("'Status Id' column not found in line status file")

print("\n✓ Required columns found")

# Display sample
line_status_df.head()

## Step 3: Verify and Display Status Values

Check what status values exist in the data.

In [ ]:
# Check unique status IDs
unique_statuses = sorted(line_status_df['Status Id'].dropna().unique())
print(f"Unique Status IDs found: {unique_statuses}")
print(f"Number of unique statuses: {len(unique_statuses)}")

# Show distribution
print("\nStatus distribution:")
if 'Status' in line_status_df.columns:
    status_dist = line_status_df.groupby(['Status Id', 'Status']).size().reset_index(name='Count')
    display(status_dist)
else:
    status_counts = line_status_df['Status Id'].value_counts().sort_index()
    display(pd.DataFrame({'Status Id': status_counts.index, 'Count': status_counts.values}))

## Step 4: Extract and Parse Line Names from Line_Unique

Parse the `Line_Unique` column which has complex string format like:
- `["['rail_1_1' 'rail_1_2']"]`
- `['line1', 'line2']`
- `line1, line2`

Create a separate dataframe with (group, line) pairs.

In [ ]:
def parse_line_unique_robust(line_str):
    """
    Robustly parse Line_Unique column to extract individual line names.
    
    Handles various formats:
    - "['line1' 'line2']"
    - "['line1', 'line2']"
    - '[line1, line2]'
    - 'line1, line2'
    - Space-separated: 'line1 line2'
    
    Args:
        line_str: String representation of line list
    
    Returns:
        List of individual line names
    """
    if pd.isna(line_str) or line_str == '':
        return []
    
    line_str = str(line_str).strip()
    
    # Remove outer brackets and quotes
    line_str = line_str.strip('[]"\'')
    
    # Check if it's a nested structure like "['item1' 'item2']"
    # This pattern catches space-separated values within quotes
    nested_pattern = r"\[([^\]]+)\]"
    nested_match = re.search(nested_pattern, line_str)
    if nested_match:
        inner_content = nested_match.group(1)
        # Remove all quotes
        inner_content = inner_content.replace("'", "").replace('"', '')
        # Split by space or comma
        lines = re.split(r'[\s,]+', inner_content)
        lines = [x.strip() for x in lines if x.strip()]
        return lines
    
    # Remove all remaining quotes
    line_str = line_str.replace("'", "").replace('"', '')
    
    # Try splitting by comma first
    if ',' in line_str:
        lines = [x.strip() for x in line_str.split(',') if x.strip()]
    # Then try space-separated
    elif ' ' in line_str:
        lines = [x.strip() for x in line_str.split() if x.strip()]
    # Single line
    else:
        lines = [line_str] if line_str else []
    
    # Final cleanup: remove any remaining brackets
    lines = [x.strip('[]') for x in lines if x.strip('[]')]
    
    return lines

# Test the parser with sample data
print("Testing parser with sample Line_Unique values:")
sample_values = hubs_complete_df['Line_Unique'].head(5)
for val in sample_values:
    parsed = parse_line_unique_robust(val)
    print(f"Input: {val[:100]}..." if len(str(val)) > 100 else f"Input: {val}")
    print(f"Parsed: {parsed}")
    print(f"Count: {len(parsed)}")
    print("-" * 60)

## Step 5: Create Group-Line Mapping DataFrame

Extract (group, Line_Unique) and parse to create exploded (group, line) pairs.

In [ ]:
# Extract group and Line_Unique columns
print("Extracting group and Line_Unique columns...")
group_lines_df = hubs_complete_df[['group', 'Line_Unique']].copy()

print(f"Starting with {len(group_lines_df)} hubs")

# Parse Line_Unique to extract individual lines
print("\nParsing Line_Unique column...")
group_lines_df['lines_list'] = group_lines_df['Line_Unique'].apply(parse_line_unique_robust)

# Count total lines before explosion
total_lines_before = group_lines_df['lines_list'].apply(len).sum()
print(f"Total lines found: {total_lines_before}")

# Show distribution of line counts per hub
lines_per_hub = group_lines_df['lines_list'].apply(len)
print(f"\nLines per hub statistics:")
print(f"  Min: {lines_per_hub.min()}")
print(f"  Max: {lines_per_hub.max()}")
print(f"  Mean: {lines_per_hub.mean():.2f}")
print(f"  Median: {lines_per_hub.median():.0f}")

# Display sample
print("\nSample of parsed data:")
display(group_lines_df[['group', 'lines_list']].head(10))

## Step 6: Explode to Create One Row per Line

Transform the dataframe so each line gets its own row with its group.

In [ ]:
# Explode the lines list so each line gets its own row
print("Exploding to create one row per (group, line) pair...")

group_line_pairs = group_lines_df.explode('lines_list').copy()

# Rename column for clarity
group_line_pairs.rename(columns={'lines_list': 'LineName'}, inplace=True)

# Remove rows with empty line names
group_line_pairs = group_line_pairs[group_line_pairs['LineName'].notna()].copy()
group_line_pairs = group_line_pairs[group_line_pairs['LineName'] != ''].copy()

# Drop the original Line_Unique column
group_line_pairs = group_line_pairs[['group', 'LineName']].copy()

print(f"\nCreated {len(group_line_pairs)} (group, line) pairs")
print(f"Unique groups: {group_line_pairs['group'].nunique()}")
print(f"Unique lines: {group_line_pairs['LineName'].nunique()}")

# Display sample
print("\nSample of exploded data:")
display(group_line_pairs.head(15))

# Show most common lines
print("\nTop 10 most common lines:")
top_lines = group_line_pairs['LineName'].value_counts().head(10)
display(pd.DataFrame({'LineName': top_lines.index, 'Count (hubs)': top_lines.values}))

## Step 7: Join with Line Status Data

Add Status Id to each (group, line) pair by joining with the line status data.

In [ ]:
# Join group-line pairs with status data
print("Joining with line status data...")

# Ensure LineName columns are strings and stripped
group_line_pairs['LineName'] = group_line_pairs['LineName'].astype(str).str.strip()
line_status_df['LineName'] = line_status_df['LineName'].astype(str).str.strip()

# Merge on LineName
group_line_status = group_line_pairs.merge(
    line_status_df[['LineName', 'Status Id']],
    on='LineName',
    how='left'
)

print(f"\nAfter join: {len(group_line_status)} records")

# Check for unmatched lines
unmatched = group_line_status['Status Id'].isna().sum()
if unmatched > 0:
    print(f"\n⚠️  WARNING: {unmatched} line-group pairs have no status information")
    print(f"   This represents {unmatched / len(group_line_status) * 100:.1f}% of all pairs")
    
    # Show some unmatched lines
    unmatched_lines = group_line_status[group_line_status['Status Id'].isna()]['LineName'].unique()
    print(f"\n   Sample of unmatched lines (first 20):")
    for line in unmatched_lines[:20]:
        print(f"     - {line}")
    
    # Ask whether to drop or keep
    print("\n   These will be excluded from status counts.")
else:
    print("\n✓ All lines matched successfully!")

# Display sample with status
print("\nSample of joined data:")
display(group_line_status.head(15))

# Show status distribution
print("\nStatus distribution in joined data:")
status_dist = group_line_status['Status Id'].value_counts().sort_index()
display(pd.DataFrame({'Status Id': status_dist.index, 'Count': status_dist.values}))

## Step 8: Aggregate Line Counts by Group and Status

For each group (hub), count how many lines have each status (0-7).

In [ ]:
# Remove records with no status (if any)
group_line_status_clean = group_line_status[group_line_status['Status Id'].notna()].copy()

print(f"Aggregating line counts by group and status...")
print(f"Working with {len(group_line_status_clean)} records")

# Group by group and Status Id, count lines
status_counts = group_line_status_clean.groupby(['group', 'Status Id']).size().reset_index(name='count')

print(f"\nAggregated to {len(status_counts)} (group, status) combinations")

# Pivot to wide format: one row per group, one column per status
status_wide = status_counts.pivot(index='group', columns='Status Id', values='count')

# Rename columns to NumLinesStatus_X
status_wide.columns = [f'NumLinesStatus_{int(col)}' for col in status_wide.columns]

# Reset index to make 'group' a column
status_wide = status_wide.reset_index()

# Fill NaN with 0 (groups with no lines in that status)
status_cols = [col for col in status_wide.columns if col.startswith('NumLinesStatus_')]
status_wide[status_cols] = status_wide[status_cols].fillna(0).astype(int)

print(f"\nAggregated status counts for {len(status_wide)} groups")
print(f"Status columns created: {status_cols}")

# Display sample
print("\nSample of aggregated data:")
display(status_wide.head(10))

## Step 9: Ensure All Status Columns (0-7) Exist

Add any missing status columns and set them to 0.

In [ ]:
# Ensure all status columns 0-7 exist
expected_status_cols = [f'NumLinesStatus_{i}' for i in range(8)]

for col in expected_status_cols:
    if col not in status_wide.columns:
        print(f"Adding missing column: {col}")
        status_wide[col] = 0

# Reorder columns: group first, then status columns in order
status_cols_ordered = sorted([col for col in status_wide.columns if col.startswith('NumLinesStatus_')])
status_wide = status_wide[['group'] + status_cols_ordered]

print(f"\nFinal status columns: {status_cols_ordered}")
print(f"Total groups with status data: {len(status_wide)}")

# Display sample
display(status_wide.head(10))

## Step 10: Load Final Scored Hubs Data

Load `hubs_scored_final.csv` to which we'll add the status columns.

In [ ]:
# Load final scored hubs
hubs_scored_file = RESULTS_DIR / 'hubs_scored_final.csv'

# Check alternative locations if not found
if not hubs_scored_file.exists():
    alternative_paths = [
        PROCESSED_DATA_DIR / 'hubs_scored_final.csv',
        DATA_DIR / 'hubs_scored_final.csv',
        RESULTS_DIR / 'scored_hubs_final.csv',
        # Try to find any scored hubs file
        *sorted(RESULTS_DIR.glob('scored_hubs_*.csv')),
        *sorted(RESULTS_DIR.glob('hubs_*_final.csv')),
    ]
    
    for path in alternative_paths:
        if path.exists():
            hubs_scored_file = path
            break

if not hubs_scored_file.exists():
    raise FileNotFoundError(
        f"Could not find 'hubs_scored_final.csv'.\n"
        f"Please ensure this file exists in one of these locations:\n"
        f"  - {RESULTS_DIR}\n"
        f"  - {PROCESSED_DATA_DIR}\n"
        f"  - {DATA_DIR}"
    )

print(f"Loading final scored hubs from: {hubs_scored_file.name}")
hubs_scored_df = pd.read_csv(hubs_scored_file, encoding='utf-8')

print(f"\nLoaded {len(hubs_scored_df)} scored hubs")
print(f"Columns: {len(hubs_scored_df.columns)}")

# Check for group column
if 'group' not in hubs_scored_df.columns:
    raise ValueError("'group' column not found in hubs_scored_final.csv")

print("\n✓ 'group' column found")

# Display sample
print("\nSample of scored hubs:")
display_cols = ['group']
for col in ['HubName', 'HubType', 'Overall_Rank', 'Average_Simulated_Score', 'TotalDemand']:
    if col in hubs_scored_df.columns:
        display_cols.append(col)

display(hubs_scored_df[display_cols].head())

## Step 11: Merge Status Columns into Final Scored Hubs

Add the status count columns to the final scored hubs dataframe.

In [ ]:
# Merge status counts with scored hubs
print(f"Merging status counts into scored hubs...")
print(f"Hubs before merge: {len(hubs_scored_df)}")

hubs_with_status = hubs_scored_df.merge(
    status_wide,
    on='group',
    how='left'
)

print(f"Hubs after merge: {len(hubs_with_status)}")

# Fill NaN with 0 for hubs that don't have any lines in status data
status_cols = [col for col in hubs_with_status.columns if col.startswith('NumLinesStatus_')]
hubs_with_status[status_cols] = hubs_with_status[status_cols].fillna(0).astype(int)

print(f"\n✓ Status columns added: {len(status_cols)}")
print(f"Total columns now: {len(hubs_with_status.columns)}")

# Count hubs with status data
hubs_with_any_status = (hubs_with_status[status_cols].sum(axis=1) > 0).sum()
print(f"\nHubs with at least one line status: {hubs_with_any_status} / {len(hubs_with_status)}")

# Display sample with new columns
print("\nSample with status columns:")
display_cols = ['group']
if 'HubType' in hubs_with_status.columns:
    display_cols.append('HubType')
if 'TotalDemand' in hubs_with_status.columns:
    display_cols.append('TotalDemand')
display_cols.extend(status_cols)

display(hubs_with_status[display_cols].head(10))

## Step 12: Add Total Lines Column

Add a column with the total count of all lines across all statuses.

In [ ]:
# Calculate total lines across all statuses
hubs_with_status['TotalLinesAllStatuses'] = hubs_with_status[status_cols].sum(axis=1)

print(f"✓ Total lines column added")
print(f"\nSummary statistics for total lines:")
print(hubs_with_status['TotalLinesAllStatuses'].describe())

# Show hubs with most lines
print(f"\n\nTop 10 hubs by total line count:")
top_by_lines = hubs_with_status.nlargest(10, 'TotalLinesAllStatuses')

display_cols = ['group', 'TotalLinesAllStatuses'] + status_cols
if 'HubName' in hubs_with_status.columns:
    display_cols.insert(1, 'HubName')
if 'address' in hubs_with_status.columns and 'HubName' not in hubs_with_status.columns:
    display_cols.insert(1, 'address')

display(top_by_lines[display_cols])

## Step 13: Summary Statistics by Status

Display distribution of lines by status across all hubs.

In [ ]:
# Calculate summary statistics for each status
print("Summary Statistics by Status")
print("=" * 80)

summary_data = []
for col in status_cols:
    status_num = col.replace('NumLinesStatus_', '')
    
    total_lines = hubs_with_status[col].sum()
    hubs_with_lines = (hubs_with_status[col] > 0).sum()
    max_lines = hubs_with_status[col].max()
    avg_all = hubs_with_status[col].mean()
    avg_nonzero = hubs_with_status[hubs_with_status[col] > 0][col].mean() if hubs_with_lines > 0 else 0
    
    # Get status name if available
    status_name = ''
    if 'Status' in line_status_df.columns:
        matching_status = line_status_df[line_status_df['Status Id'] == int(status_num)]['Status'].unique()
        if len(matching_status) > 0:
            status_name = matching_status[0]
    
    summary = {
        'Status ID': status_num,
        'Status Name': status_name,
        'Total Lines': total_lines,
        'Hubs with Lines': hubs_with_lines,
        'Max Lines (one hub)': max_lines,
        'Avg per Hub (all)': f"{avg_all:.2f}",
        'Avg per Hub (non-zero)': f"{avg_nonzero:.2f}"
    }
    summary_data.append(summary)

summary_df = pd.DataFrame(summary_data)
display(summary_df)

# Overall totals
print(f"\n{'=' * 80}")
print(f"Total lines across all hubs and statuses: {hubs_with_status['TotalLinesAllStatuses'].sum()}")
print(f"Average lines per hub: {hubs_with_status['TotalLinesAllStatuses'].mean():.2f}")
print(f"Hubs with at least one line: {(hubs_with_status['TotalLinesAllStatuses'] > 0).sum()} / {len(hubs_with_status)}")

## Step 14: Export Results

Save the enriched scored hubs data with line status columns.

In [ ]:
# Create output filename with timestamp
timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M')
output_file = RESULTS_DIR / f"hubs_scored_with_line_status_{timestamp}.csv"

# Reorder columns to put new status columns near the beginning
# Priority: group, hub info, scores, status columns, then everything else
priority_cols = ['group']

if 'HubName' in hubs_with_status.columns:
    priority_cols.append('HubName')
if 'HubType' in hubs_with_status.columns:
    priority_cols.append('HubType')
if 'Overall_Rank' in hubs_with_status.columns:
    priority_cols.append('Overall_Rank')
if 'Average_Simulated_Score' in hubs_with_status.columns:
    priority_cols.append('Average_Simulated_Score')
if 'TotalDemand' in hubs_with_status.columns:
    priority_cols.append('TotalDemand')

# Add total lines and status columns
priority_cols.append('TotalLinesAllStatuses')
priority_cols.extend(status_cols)

# Add remaining columns
other_cols = [col for col in hubs_with_status.columns if col not in priority_cols]
final_col_order = [col for col in priority_cols if col in hubs_with_status.columns] + other_cols

# Reorder and export
hubs_export = hubs_with_status[final_col_order].copy()

# Sort by rank if available, otherwise by group
if 'Overall_Rank' in hubs_export.columns:
    hubs_export = hubs_export.sort_values('Overall_Rank')
else:
    hubs_export = hubs_export.sort_values('group')

# Save to CSV
hubs_export.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"✓ Results exported to: {output_file.name}")
print(f"\nFull path: {output_file}")
print(f"\nExport summary:")
print(f"  Total hubs: {len(hubs_export)}")
print(f"  Total columns: {len(hubs_export.columns)}")
print(f"\nNew columns added:")
print(f"  - TotalLinesAllStatuses")
for col in status_cols:
    print(f"  - {col}")

## Summary

Post-processing complete! ✓

### Process Summary:

1. **Loaded data:**
   - `hubs_with_complete_data.csv` (line names in `Line_Unique`)
   - `hubs_scored_final.csv` (final scored hubs)
   - Line status CSV (LineName, Status Id)

2. **Parsed Line_Unique:**
   - Extracted individual line names from complex string format
   - Created (group, line) pairs

3. **Joined with status:**
   - Matched lines to status data
   - Aggregated by group and Status Id

4. **Added to final output:**
   - 9 new columns: `NumLinesStatus_0` through `NumLinesStatus_7` + `TotalLinesAllStatuses`
   - Each counts lines in that hub with corresponding status

### Output File:
- Saved to: `data/results/hubs_scored_with_line_status_YYYYMMDD_HHMM.csv`
- Contains all original columns from `hubs_scored_final.csv` plus new status columns

### Next Steps:
- Review the output file to verify results
- Use status columns for analysis or visualization
- Consider visualizations (e.g., stacked bar charts by hub type and status)